# From Monte Carlo to Critical Exponents

This notebook demonstrates finite-size scaling analysis:
1. Run MCMC at multiple system sizes and temperatures
2. Compute the Binder cumulant at each (N, beta)
3. Find the Binder crossing to estimate beta_c
4. Perform data collapse to extract the exponent nu
5. Compare with the mean-field prediction nu = 5/2

**Runtime:** < 15 min on CPU

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from parisijax.analysis.scaling import (
    collect_observables,
    data_collapse,
    find_binder_crossing,
)

## 1. Collect Observables at Multiple (N, beta) Points

We use small system sizes for speed. Larger sizes give better scaling but take longer.

In [ ]:
sizes = [16, 32, 64]
betas = jnp.linspace(0.5, 1.8, 12)

key = jax.random.PRNGKey(42)
obs = collect_observables(
    key, sizes, betas,
    n_samples=100,
    n_steps=2000,
    burnin=500,
)

print(f"Collected {len(obs)} data points")
print(f"Example: N=32, beta=1.0 -> g={obs[(32, float(betas[4]))][ 'g']:.4f}")

## 2. Binder Cumulant vs Temperature

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

betas_np = np.array(betas)
for n_spins in sizes:
    g_vals = [obs[(n_spins, float(b))]['g'] for b in betas]
    ax.plot(betas_np, g_vals, 'o-', label=f'N={n_spins}', markersize=5)

ax.axvline(1.0, color='red', ls=':', alpha=0.5, label=r'$\beta_c = 1$ (theory)')
ax.set_xlabel(r'Inverse temperature $\beta$')
ax.set_ylabel('Binder cumulant $g$')
ax.set_title('Binder Cumulant: Crossing at T_c')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 3. Binder Crossing

In [ ]:
beta_c_crossing = find_binder_crossing(obs, sizes, betas)
print(f"Binder crossing estimate: beta_c = {beta_c_crossing:.4f}")
print("Theory: beta_c = 1.0")
print(f"Error: {abs(beta_c_crossing - 1.0):.4f}")

## 4. Susceptibility

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for n_spins in sizes:
    chi_vals = [obs[(n_spins, float(b))]['chi'] for b in betas]
    ax.plot(betas_np, chi_vals, 'o-', label=f'N={n_spins}', markersize=5)

ax.axvline(1.0, color='red', ls=':', alpha=0.5)
ax.set_xlabel(r'$\beta$')
ax.set_ylabel(r'$\chi_{SG} = N \langle q^2 \rangle$')
ax.set_title('Spin Glass Susceptibility')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 5. Data Collapse

In [ ]:
scaling_result = data_collapse(obs, sizes, betas, observable_key='g')

print("Data collapse results:")
print(f"  beta_c = {scaling_result.beta_c:.4f} (theory: 1.0)")
print(f"  nu     = {scaling_result.nu:.4f} (mean-field: 2.5)")
print(f"  quality = {scaling_result.quality:.6f}")

## 6. Scaled Data Collapse Plot

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Unscaled
for n_spins in sizes:
    g_vals = [obs[(n_spins, float(b))]['g'] for b in betas]
    ax1.plot(betas_np, g_vals, 'o-', label=f'N={n_spins}', markersize=5)
ax1.set_xlabel(r'$\beta$')
ax1.set_ylabel('$g$')
ax1.set_title('Unscaled')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Scaled (data collapse)
beta_c = scaling_result.beta_c
nu = scaling_result.nu
for n_spins in sizes:
    g_vals = [obs[(n_spins, float(b))]['g'] for b in betas]
    x_scaled = (n_spins ** (1.0 / (3.0 * nu))) * (betas_np - beta_c)
    ax2.plot(x_scaled, g_vals, 'o-', label=f'N={n_spins}', markersize=5)
ax2.set_xlabel(r'$N^{1/3\nu}(\beta - \beta_c)$')
ax2.set_ylabel('$g$')
ax2.set_title(f'Data Collapse ($\\beta_c$={beta_c:.3f}, $\\nu$={nu:.2f})')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()